# GPU Practice: Fine-Tune and Evaluate Every Riverside Novel

> **The mission:** run the same leakage-resistant LoRA continued-pretraining experiment independently on all eight Riverside novels and accept only evidence measured on untouched chapters.

This fourth notebook turns the techniques from Parts 1-3 into one controlled practice run. It has no CPU fallback: a CUDA GPU is part of the experiment contract. Every novel starts from the same pinned SmolLM2 base revision, uses the same hyperparameters, and writes an independent adapter and manifest.

## Evidence contract

- Split complete chapters before tokenization: early chapters train, the next block validates checkpoints, and the final block is tested once after selection.
- Create chunks within one chapter only; no example crosses a chapter or split boundary.
- Select checkpoints only by explicit token-weighted validation NLL.
- Compare the selected adapter with its unchanged base on test perplexity, chapter-resampled uncertainty, general-language retention, and matched deterministic generations.
- Treat generations as inspection evidence, not as the selection metric.

The full eight-run experiment can take hours depending on GPU throughput. A result is evidence only after every cell completes on CUDA and all eight manifests are written.

## 1. Configure the GPU experiment

The model revision and recipe are fixed before any held-out result is visible. `OVERWRITE_RUNS` is intentionally separate from `RUN_TRAINING`: rerunning a completed novel must be an explicit destructive choice, and only that novel's output directory may be replaced.

The 360M base is small enough for ordinary LoRA on a CUDA device, so this notebook does not use QLoRA or present inference quantization as training.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import random
import shutil
import time
from contextlib import nullcontext
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        content_dir = candidate / "learning" / "genai" / "content"
        chapter_notebook = (
            candidate
            / "learning"
            / "genai"
            / "09-llm-finetuning"
            / "01-llm-finetuning-data-techniques.ipynb"
        )
        if content_dir.is_dir() and chapter_notebook.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repository or a descendant directory; "
        "the shared corpus is expected under learning/genai/content."
    )


NOTEBOOK_DIR = find_repo_root()  # Retained name: manifests store repository-relative source paths.
CHAPTER_DIR = NOTEBOOK_DIR / "learning" / "genai" / "09-llm-finetuning"
CONTENT_DIR = NOTEBOOK_DIR / "learning" / "genai" / "content"
ARTIFACT_ROOT = CHAPTER_DIR / "checkpoints" / "all-novel-gpu"

RUN_TRAINING = False  # Explicit opt-in: the complete eight-novel run can take hours.
OVERWRITE_RUNS = False
BASE_SEED = 20260804
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
MODEL_REVISION = "f8027fd0eaeea54caa13c31d31b9fdc459c38b49"
MAX_LENGTH = 512
MIN_TOKENS = 64
MAX_STEPS = 100
SAVE_STEPS = 25
BOOTSTRAP_REPLICATES = 2_000
GENERAL_RETENTION_LIMIT_PCT = 5.0
DOMAIN_IMPROVEMENT_TARGET_PCT = 5.0

NOVELS = {
    "the-weight-of-distant-light": "The Weight of Distant Light",
    "the-tidebound-accord": "The Tidebound Accord",
    "the-cartographers-cipher": "The Cartographer's Cipher",
    "the-silk-merchants-daughter": "The Silk Merchant's Daughter",
    "neural-drift": "Neural Drift",
    "the-hollow-beneath": "The Hollow Beneath",
    "the-weight-of-tides": "The Weight of Tides",
    "the-everglades-cipher": "The Everglades Cipher",
}

GENERAL_RETENTION_TEXTS = [
    "Water changes state when heat alters the motion of its molecules. At standard pressure, ice melts into liquid water and liquid water boils into vapor.",
    "A constitutional government divides public authority among institutions and defines procedures for making, applying, and reviewing laws.",
    "Photosynthesis converts light energy into chemical energy. Plants use carbon dioxide and water to produce sugars while releasing oxygen.",
    "A database index stores an auxiliary structure that helps a query locate rows without scanning every record in a table.",
    "The Pacific Ocean is Earth's largest ocean basin. It lies between Asia and Australia to the west and the Americas to the east.",
    "Musical rhythm organizes sounds and silences through time. Meter groups recurring beats, while tempo describes how quickly those beats proceed.",
]
GENERAL_RETENTION_TEXT = "\n\n".join(GENERAL_RETENTION_TEXTS)
GENERAL_RETENTION_SHA256 = hashlib.sha256(GENERAL_RETENTION_TEXT.encode("utf-8")).hexdigest()

if not torch.cuda.is_available():
    raise RuntimeError("This practice notebook is GPU-only and requires a CUDA device visible to PyTorch.")

torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0")
GPU_PROPERTIES = torch.cuda.get_device_properties(DEVICE)
USE_BF16 = bool(torch.cuda.is_bf16_supported())
TRAIN_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
torch.cuda.manual_seed_all(BASE_SEED)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

PACKAGE_VERSIONS = {
    package: version(package)
    for package in ["torch", "transformers", "datasets", "accelerate", "peft"]
}
HARDWARE = {
    "device": str(DEVICE),
    "gpu_name": GPU_PROPERTIES.name,
    "compute_capability": f"{GPU_PROPERTIES.major}.{GPU_PROPERTIES.minor}",
    "total_vram_bytes": GPU_PROPERTIES.total_memory,
    "cuda_runtime": torch.version.cuda,
    "mixed_precision": "bf16" if USE_BF16 else "fp16",
}

print(f"Repository root: {NOTEBOOK_DIR}")
print(f"Shared content: {CONTENT_DIR}")
print(f"Chapter artifacts: {ARTIFACT_ROOT}")
print(f"GPU: {HARDWARE['gpu_name']} ({HARDWARE['compute_capability']}, {GPU_PROPERTIES.total_memory / 1024**3:.1f} GiB)")
print(f"CUDA / precision: {torch.version.cuda} / {HARDWARE['mixed_precision']}")
print(f"Model: {MODEL_NAME}@{MODEL_REVISION[:12]}")
print(PACKAGE_VERSIONS)


## 2. Reserve complete chapters

For each novel, reserve 15% of its chapters for validation and another 15% for testing. Round each holdout upward, and never use fewer than three chapters in either holdout.

| Novel length | Holdout calculation | Chronological split |
| ---: | --- | --- |
| 12 chapters | 15% rounds up to 2, so use the minimum of 3 | 6 train, 3 validation, 3 test |
| 40 chapters | 15% is 6 | 28 train, 6 validation, 6 test |
| 80 chapters | 15% is 12 | 56 train, 12 validation, 12 test |

The earliest chapters train, the next holdout selects checkpoints, and the final holdout remains untouched until testing. This chronological split is intentionally harder than a random chunk split: it measures transfer into a later narrative region and prevents adjacent passages from landing on opposite sides of a boundary. It is not an IID sample of the novel.

Only `chapter-*.txt` files are eligible. The ten Everglades appendices are reference material with a different document function, so they are inventoried as excluded rather than mixed into chapter evaluation. SHA-256 computation is the only pre-selection operation over test bytes; test text is not decoded or tokenized until a checkpoint has been selected.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def build_split(slug: str) -> dict:
    novel_dir = CONTENT_DIR / slug
    if not novel_dir.is_dir():
        raise FileNotFoundError(f"Missing required novel directory: {novel_dir}")

    chapter_files = sorted(novel_dir.glob("chapter-*.txt"))
    excluded_files = sorted(novel_dir.glob("appendix-*.txt"))
    if not chapter_files or any(path.stat().st_size == 0 for path in chapter_files):
        raise ValueError(f"{slug} has missing or empty chapter files")

    chapter_count = len(chapter_files)
    holdout_count = max(3, math.ceil(0.15 * chapter_count))
    train_end = chapter_count - 2 * holdout_count
    if train_end <= 0:
        raise ValueError(f"{slug} has too few chapters for train/validation/test splitting")

    train_files = chapter_files[:train_end]
    validation_files = chapter_files[train_end : train_end + holdout_count]
    test_files = chapter_files[train_end + holdout_count :]
    split_sets = [set(train_files), set(validation_files), set(test_files)]
    assert all(left.isdisjoint(right) for index, left in enumerate(split_sets) for right in split_sets[index + 1 :])
    assert train_files + validation_files + test_files == chapter_files

    file_hashes = {path.name: sha256_file(path) for path in chapter_files}
    return {
        "slug": slug,
        "title": NOVELS[slug],
        "train": train_files,
        "validation": validation_files,
        "test": test_files,
        "excluded": excluded_files,
        "hashes": file_hashes,
    }


SPLITS = {slug: build_split(slug) for slug in NOVELS}
assert list(SPLITS) == list(NOVELS)
assert len(SPLITS) == 8

all_hash_owners = {}
for slug, split in SPLITS.items():
    for filename, digest in split["hashes"].items():
        all_hash_owners.setdefault(digest, []).append(f"{slug}/{filename}")
exact_duplicates = {digest: owners for digest, owners in all_hash_owners.items() if len(owners) > 1}
if exact_duplicates:
    raise ValueError(f"Exact duplicate chapter files would invalidate the split audit: {exact_duplicates}")

audit_rows = []
for slug, split in SPLITS.items():
    audit_rows.append(
        {
            "novel": split["title"],
            "chapters": len(split["train"]) + len(split["validation"]) + len(split["test"]),
            "train": len(split["train"]),
            "validation": len(split["validation"]),
            "test": len(split["test"]),
            "validation_files": ", ".join(path.name for path in split["validation"]),
            "test_files": ", ".join(path.name for path in split["test"]),
            "excluded_appendices": len(split["excluded"]),
        }
    )

split_audit = pd.DataFrame(audit_rows)
display(split_audit)
print(f"PASS: {sum(row['chapters'] for row in audit_rows)} chapters assigned once across eight novels; no exact duplicates.")

## 3. Tokenize without crossing boundaries

Each chapter is encoded independently and receives exactly one terminal EOS token. Blocks never cross into another chapter. Short tails below 64 tokens are omitted because a mostly padded tail would make one chapter contribute many low-information batches.

SmolLM2 uses the same token for EOS and padding. The collator therefore masks positions where `attention_mask == 0`; masking by token ID would incorrectly remove real end-of-chapter targets.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
if tokenizer.eos_token_id is None:
    raise ValueError("The pinned tokenizer must define an EOS token")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
assert tokenizer.pad_token_id == tokenizer.eos_token_id


def records_from_texts(named_texts: dict[str, str]) -> list[dict]:
    records = []
    for source_name, text in named_texts.items():
        token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        while token_ids and token_ids[-1] == tokenizer.eos_token_id:
            token_ids.pop()
        token_ids.append(tokenizer.eos_token_id)

        for start in range(0, len(token_ids), MAX_LENGTH):
            block = token_ids[start : start + MAX_LENGTH]
            if len(block) < MIN_TOKENS:
                continue
            records.append(
                {
                    "input_ids": block,
                    "attention_mask": [1] * len(block),
                    "chapter": source_name,
                }
            )
    if not records:
        raise ValueError("Tokenization produced no blocks")
    return records


def records_from_files(paths: list[Path]) -> list[dict]:
    return records_from_texts({path.name: path.read_text(encoding="utf-8") for path in paths})


def trainer_dataset(records: list[dict]) -> Dataset:
    return Dataset.from_list(
        [
            {"input_ids": record["input_ids"], "attention_mask": record["attention_mask"]}
            for record in records
        ]
    )


class CausalCollator:
    def __call__(self, features: list[dict]) -> dict[str, torch.Tensor]:
        model_features = [
            {"input_ids": feature["input_ids"], "attention_mask": feature["attention_mask"]}
            for feature in features
        ]
        batch = tokenizer.pad(model_features, padding=True, return_tensors="pt")
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch


causal_collator = CausalCollator()


def load_base_model(*, training: bool = False):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        torch_dtype=TRAIN_DTYPE,
    ).to(DEVICE)
    model.config.use_cache = not training
    if training:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    devices = {parameter.device.type for parameter in model.parameters()}
    if devices != {"cuda"}:
        raise RuntimeError(f"GPU-only contract violated; model parameter devices: {devices}")
    return model


def score_records(model, records: list[dict], *, batch_size: int = 2) -> dict:
    grouped = {}
    for record in records:
        grouped.setdefault(record["chapter"], []).append(record)

    chapter_stats = {}
    model.eval()
    for chapter, chapter_records in grouped.items():
        nll_sum = 0.0
        token_count = 0
        loader = DataLoader(chapter_records, batch_size=batch_size, shuffle=False, collate_fn=causal_collator)
        for batch in loader:
            batch = {name: tensor.to(DEVICE) for name, tensor in batch.items()}
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE):
                logits = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                ).logits
            shift_logits = logits[:, :-1, :].contiguous().float()
            shift_labels = batch["labels"][:, 1:].contiguous()
            nll_sum += F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
                reduction="sum",
            ).item()
            token_count += int((shift_labels != -100).sum().item())

        if token_count == 0:
            raise ValueError(f"No scored tokens for {chapter}")
        mean_nll = nll_sum / token_count
        chapter_stats[chapter] = {
            "nll_sum": nll_sum,
            "tokens": token_count,
            "mean_nll": mean_nll,
            "perplexity": math.exp(min(mean_nll, 50.0)),
        }

    total_nll = sum(item["nll_sum"] for item in chapter_stats.values())
    total_tokens = sum(item["tokens"] for item in chapter_stats.values())
    mean_nll = total_nll / total_tokens
    return {
        "nll_sum": total_nll,
        "tokens": total_tokens,
        "mean_nll": mean_nll,
        "perplexity": math.exp(min(mean_nll, 50.0)),
        "chapters": chapter_stats,
    }


print(f"Tokenizer ready: {len(tokenizer):,} tokens; pad/EOS id={tokenizer.eos_token_id}.")

## 4. Select by validation, then open the test set

`Trainer` is used only for optimization and interval checkpoint writing. Its batch-averaged `eval_loss` is not the selection metric. Each saved adapter is reloaded on a fresh pinned base and scored with the token-weighted helper above; the adapter with minimum validation NLL wins.

Only after that choice is fixed does the run decode and tokenize test chapters. The final uncertainty interval resamples whole chapters with replacement. With only four to six test chapters per novel, this is a small-sample stability estimate, not a population confidence guarantee.

#### Watch validation choose the checkpoint

![Animation tracing interval checkpoints from training through validation scoring, best-checkpoint selection, and a single final test evaluation](images/validation-checkpoint-selection.gif)

Training may produce many candidate checkpoints, but the validation split is the only lane allowed to choose among them. Once the minimum validation NLL fixes the winner, the test chapters are opened once to estimate final performance. Repeatedly consulting test results would quietly turn the test set into another validation set.

In [ ]:
def make_lora_config() -> LoraConfig:
    return LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type=TaskType.CAUSAL_LM,
    )


def checkpoint_step(path: Path) -> int:
    return int(path.name.rsplit("-", 1)[-1])


def validate_checkpoints(checkpoint_root: Path, validation_records: list[dict]) -> tuple[Path, list[dict], dict]:
    checkpoints = sorted(checkpoint_root.glob("checkpoint-*"), key=checkpoint_step)
    if not checkpoints:
        raise FileNotFoundError(f"No interval checkpoints found under {checkpoint_root}")

    scores = []
    disabled_base_validation = None
    for checkpoint in checkpoints:
        base_model = load_base_model()
        candidate = PeftModel.from_pretrained(base_model, checkpoint, is_trainable=False).to(DEVICE)
        candidate.config.use_cache = True
        candidate_score = score_records(candidate, validation_records)
        if disabled_base_validation is None:
            with candidate.disable_adapter():
                disabled_base_validation = score_records(candidate, validation_records)
        scores.append(
            {
                "checkpoint": checkpoint.name,
                "step": checkpoint_step(checkpoint),
                "validation_mean_nll": candidate_score["mean_nll"],
                "validation_perplexity": candidate_score["perplexity"],
            }
        )
        del candidate, base_model
        gc.collect()
        torch.cuda.empty_cache()

    best = min(scores, key=lambda item: item["validation_mean_nll"])
    return checkpoint_root / best["checkpoint"], scores, disabled_base_validation


def aggregate_sample(chapter_stats: dict, sampled_chapters: np.ndarray) -> float:
    nll_sum = sum(chapter_stats[chapter]["nll_sum"] for chapter in sampled_chapters)
    token_count = sum(chapter_stats[chapter]["tokens"] for chapter in sampled_chapters)
    return math.exp(min(nll_sum / token_count, 50.0))


def paired_chapter_bootstrap(base_score: dict, trained_score: dict, seed: int) -> dict:
    chapters = sorted(base_score["chapters"])
    if chapters != sorted(trained_score["chapters"]):
        raise ValueError("Base and adapter chapter suites do not match")

    rng = np.random.default_rng(seed)
    improvements = np.empty(BOOTSTRAP_REPLICATES, dtype=np.float64)
    chapter_array = np.array(chapters)
    for replicate in range(BOOTSTRAP_REPLICATES):
        sampled = rng.choice(chapter_array, size=len(chapters), replace=True)
        base_ppl = aggregate_sample(base_score["chapters"], sampled)
        trained_ppl = aggregate_sample(trained_score["chapters"], sampled)
        improvements[replicate] = 100.0 * (base_ppl - trained_ppl) / base_ppl

    lower, upper = np.percentile(improvements, [2.5, 97.5])
    return {
        "seed": seed,
        "replicates": BOOTSTRAP_REPLICATES,
        "unit": "chapter",
        "chapter_count": len(chapters),
        "improvement_pct_ci95": [float(lower), float(upper)],
    }


def prompt_from_chapter(path: Path, prompt_tokens: int = 96) -> str:
    token_ids = tokenizer(path.read_text(encoding="utf-8"), add_special_tokens=False)["input_ids"]
    return tokenizer.decode(token_ids[:prompt_tokens], skip_special_tokens=True)


def generate_continuation(model, prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE):
        generated = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    continuation = generated[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(continuation, skip_special_tokens=True)


def text_sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_json(path: Path, payload: dict | list) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


print("Validation selection, chapter bootstrap, and matched-generation helpers are ready.")

## 5. Run one isolated novel experiment

Every call below creates new model, adapter, optimizer, scheduler, and checkpoint state. It never resumes implicitly. Set `OVERWRITE_RUNS = True` only when intentionally replacing existing outputs; deletion remains scoped to the current novel directory.

The status contract is conservative:

- `PASS`: test perplexity improves by at least 5%, the chapter-bootstrap lower bound is above zero, and retention regression is at most 5%.
- `FAIL`: retention regresses by more than 5%, or the entire chapter-bootstrap interval is below zero.
- `INCONCLUSIVE`: everything between those gates.

### Read the Runner as Nine Visible Stages

The next code cell is long because it keeps one novel's state isolated and records the complete evidence package. Do not memorize it line by line. Navigate it by these stage boundaries:

```mermaid
flowchart LR
    Split["1. Load frozen split"] --> Records["2. Build chapter-bounded records"]
    Records --> Train["3. Train interval checkpoints"]
    Train --> Select["4. Select on validation NLL"]
    Select --> Reload["5. Save and verify reload parity"]
    Reload --> Test["6. Open untouched test chapters"]
    Test --> Uncertainty["7. Bootstrap chapters + check retention"]
    Uncertainty --> Gate["8. Assign PASS / FAIL / INCONCLUSIVE"]
    Gate --> Manifest["9. Write result and manifest"]
```

Each stage leaves a named object in the manifest. If a run fails, locate the last completed stage instead of debugging the complete function as one opaque unit.

In [ ]:
def split_manifest(split: dict) -> dict:
    return {
        partition: [
            {
                "path": path.relative_to(NOTEBOOK_DIR).as_posix(),
                "sha256": split["hashes"][path.name],
                "bytes": path.stat().st_size,
            }
            for path in split[partition]
        ]
        for partition in ["train", "validation", "test"]
    }


def run_novel(slug: str, novel_index: int) -> dict:
    if not RUN_TRAINING:
        raise RuntimeError("Set RUN_TRAINING=True to execute the evidence workflow.")

    split = SPLITS[slug]
    seed = BASE_SEED + novel_index
    run_dir = ARTIFACT_ROOT / slug
    checkpoint_root = run_dir / "training-checkpoints"
    selected_adapter_dir = run_dir / "selected-adapter"

    if run_dir.exists():
        if not OVERWRITE_RUNS:
            raise FileExistsError(
                f"{run_dir} already exists. Inspect it or set OVERWRITE_RUNS=True to replace this run explicitly."
            )
        shutil.rmtree(run_dir)
    checkpoint_root.mkdir(parents=True)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    torch.cuda.synchronize(DEVICE)
    run_started = time.perf_counter()

    trainer = None
    training_base = None
    training_model = None
    selected_base = None
    selected_model = None
    evaluation_base = None
    evaluation_model = None

    try:
        train_records = records_from_files(split["train"])
        validation_records = records_from_files(split["validation"])
        train_data = trainer_dataset(train_records)

        training_base = load_base_model(training=True)
        available_leaf_modules = {name.rsplit(".", 1)[-1] for name, _ in training_base.named_modules()}
        required_targets = {"q_proj", "k_proj", "v_proj", "o_proj"}
        missing_targets = required_targets - available_leaf_modules
        if missing_targets:
            raise ValueError(f"Pinned model is missing LoRA targets: {sorted(missing_targets)}")

        training_model = get_peft_model(training_base, make_lora_config()).to(DEVICE)
        training_model.config.use_cache = False
        parameter_devices = {parameter.device.type for parameter in training_model.parameters()}
        if parameter_devices != {"cuda"}:
            raise RuntimeError(f"GPU-only contract violated after PEFT wrapping: {parameter_devices}")

        training_arguments = TrainingArguments(
            output_dir=str(checkpoint_root),
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=1e-4,
            max_steps=MAX_STEPS,
            warmup_ratio=0.1,
            weight_decay=0.01,
            max_grad_norm=1.0,
            logging_strategy="steps",
            logging_steps=5,
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=MAX_STEPS // SAVE_STEPS,
            bf16=USE_BF16,
            fp16=not USE_BF16,
            gradient_checkpointing=True,
            optim="adamw_torch",
            report_to="none",
            remove_unused_columns=False,
            dataloader_pin_memory=True,
            seed=seed,
            data_seed=seed,
        )
        trainer = Trainer(
            model=training_model,
            args=training_arguments,
            train_dataset=train_data,
            data_collator=causal_collator,
        )
        trainer.train()
        training_history = list(trainer.state.log_history)

        trainer = None
        training_model = None
        training_base = None
        train_data = None
        gc.collect()
        torch.cuda.empty_cache()

        selected_checkpoint, checkpoint_scores, base_validation = validate_checkpoints(
            checkpoint_root,
            validation_records,
        )

        selected_base = load_base_model()
        selected_model = PeftModel.from_pretrained(
            selected_base,
            selected_checkpoint,
            is_trainable=False,
        ).to(DEVICE)
        selected_model.config.use_cache = True
        selected_validation = score_records(selected_model, validation_records)
        selected_model.save_pretrained(selected_adapter_dir, safe_serialization=True)
        tokenizer.save_pretrained(selected_adapter_dir)

        selected_model = None
        selected_base = None
        gc.collect()
        torch.cuda.empty_cache()

        evaluation_base = load_base_model()
        evaluation_model = PeftModel.from_pretrained(
            evaluation_base,
            selected_adapter_dir,
            is_trainable=False,
        ).to(DEVICE)
        evaluation_model.config.use_cache = True
        reload_validation = score_records(evaluation_model, validation_records)
        with evaluation_model.disable_adapter():
            reload_base_validation = score_records(evaluation_model, validation_records)

        selected_parity_delta = abs(
            reload_validation["mean_nll"] - selected_validation["mean_nll"]
        )
        base_parity_delta = abs(
            reload_base_validation["mean_nll"] - base_validation["mean_nll"]
        )
        if selected_parity_delta > 1e-5 or base_parity_delta > 1e-5:
            raise RuntimeError(
                f"Adapter reload parity failed: selected={selected_parity_delta:.3e}, base={base_parity_delta:.3e}"
            )

        # This is the first point at which test chapters are decoded or tokenized.
        test_records = records_from_files(split["test"])
        trained_test = score_records(evaluation_model, test_records)
        with evaluation_model.disable_adapter():
            base_test = score_records(evaluation_model, test_records)

        retention_records = records_from_texts({"general-retention": GENERAL_RETENTION_TEXT})
        trained_retention = score_records(evaluation_model, retention_records)
        with evaluation_model.disable_adapter():
            base_retention = score_records(evaluation_model, retention_records)

        test_improvement_pct = 100.0 * (
            base_test["perplexity"] - trained_test["perplexity"]
        ) / base_test["perplexity"]
        retention_regression_pct = 100.0 * (
            trained_retention["perplexity"] - base_retention["perplexity"]
        ) / base_retention["perplexity"]
        bootstrap = paired_chapter_bootstrap(base_test, trained_test, seed + 100_000)
        ci_lower, ci_upper = bootstrap["improvement_pct_ci95"]

        domain_gate = test_improvement_pct >= DOMAIN_IMPROVEMENT_TARGET_PCT and ci_lower > 0.0
        retention_gate = retention_regression_pct <= GENERAL_RETENTION_LIMIT_PCT
        if not retention_gate or ci_upper < 0.0:
            status = "FAIL"
        elif domain_gate:
            status = "PASS"
        else:
            status = "INCONCLUSIVE"

        generations = []
        for path in split["test"][:2]:
            prompt = prompt_from_chapter(path)
            with evaluation_model.disable_adapter():
                base_output = generate_continuation(evaluation_model, prompt)
            trained_output = generate_continuation(evaluation_model, prompt)
            generations.append(
                {
                    "source_chapter": path.name,
                    "prompt": prompt,
                    "prompt_sha256": text_sha256(prompt),
                    "base": base_output,
                    "trained": trained_output,
                }
            )

        torch.cuda.synchronize(DEVICE)
        elapsed_seconds = time.perf_counter() - run_started
        peak_gpu_bytes = torch.cuda.max_memory_allocated(DEVICE)
        selected_step = checkpoint_step(selected_checkpoint)

        result = {
            "novel": split["title"],
            "slug": slug,
            "train_chapters": len(split["train"]),
            "validation_chapters": len(split["validation"]),
            "test_chapters": len(split["test"]),
            "selected_step": selected_step,
            "test_base_perplexity": base_test["perplexity"],
            "test_trained_perplexity": trained_test["perplexity"],
            "test_improvement_pct": test_improvement_pct,
            "bootstrap_ci95_lower_pct": ci_lower,
            "bootstrap_ci95_upper_pct": ci_upper,
            "retention_regression_pct": retention_regression_pct,
            "elapsed_minutes": elapsed_seconds / 60.0,
            "peak_gpu_gib": peak_gpu_bytes / 1024**3,
            "status": status,
        }

        manifest = {
            "schema_version": 1,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "objective": "chapter-disjoint LoRA continued pretraining",
            "novel": {"slug": slug, "title": split["title"]},
            "seed": seed,
            "model": {"id": MODEL_NAME, "revision": MODEL_REVISION},
            "tokenizer": {"id": MODEL_NAME, "revision": MODEL_REVISION},
            "hardware": HARDWARE,
            "package_versions": PACKAGE_VERSIONS,
            "sources": split_manifest(split),
            "excluded_files": [path.relative_to(NOTEBOOK_DIR).as_posix() for path in split["excluded"]],
            "general_retention_sha256": GENERAL_RETENTION_SHA256,
            "tokenization": {
                "max_length": MAX_LENGTH,
                "minimum_tail_tokens": MIN_TOKENS,
                "append_one_eos_per_chapter": True,
                "train_blocks": len(train_records),
                "validation_blocks": len(validation_records),
                "test_blocks": len(test_records),
            },
            "lora": {
                "rank": 8,
                "alpha": 16,
                "dropout": 0.05,
                "bias": "none",
                "targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
            },
            "training": {
                "per_device_batch_size": 1,
                "gradient_accumulation_steps": 8,
                "learning_rate": 1e-4,
                "max_steps": MAX_STEPS,
                "warmup_ratio": 0.1,
                "weight_decay": 0.01,
                "max_grad_norm": 1.0,
                "save_steps": SAVE_STEPS,
                "mixed_precision": HARDWARE["mixed_precision"],
                "history": training_history,
            },
            "selection": {
                "metric": "explicit token-weighted validation NLL",
                "checkpoint_scores": checkpoint_scores,
                "selected_checkpoint": selected_checkpoint.name,
                "selected_step": selected_step,
                "selected_validation": selected_validation,
                "reload_validation": reload_validation,
                "selected_reload_nll_delta": selected_parity_delta,
                "disabled_base_reload_nll_delta": base_parity_delta,
            },
            "final_evaluation": {
                "test_base": base_test,
                "test_trained": trained_test,
                "test_improvement_pct": test_improvement_pct,
                "general_retention_base": base_retention,
                "general_retention_trained": trained_retention,
                "general_retention_regression_pct": retention_regression_pct,
                "bootstrap": bootstrap,
                "domain_gate": domain_gate,
                "retention_gate": retention_gate,
                "status": status,
            },
            "generations": {
                "decoding": {"do_sample": False, "max_new_tokens": 80},
                "samples": generations,
            },
            "runtime": {
                "elapsed_seconds": elapsed_seconds,
                "peak_allocated_gpu_bytes": peak_gpu_bytes,
            },
        }
        write_json(run_dir / "experiment-manifest.json", manifest)
        write_json(selected_adapter_dir / "experiment-manifest.json", manifest)
        return result
    finally:
        trainer = None
        training_model = None
        training_base = None
        selected_model = None
        selected_base = None
        evaluation_model = None
        evaluation_base = None
        gc.collect()
        torch.cuda.empty_cache()


print("Per-novel experiment runner is ready.")

## 6. Execute all eight novels

The loop writes a partial result ledger after each completed novel, so a later failure does not erase earlier evidence. A successful finish requires exactly one result and one manifest for every allowlisted novel.

In [ ]:
all_results = []
results_frame = pd.DataFrame()

if not RUN_TRAINING:
    print(
        "GPU experiment is disabled. Review the split audit, then set "
        "RUN_TRAINING=True in the configuration cell to opt in."
    )
else:
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

    for novel_index, slug in enumerate(NOVELS):
        print(f"\n{'=' * 88}\n{novel_index + 1}/8: {NOVELS[slug]}\n{'=' * 88}")
        novel_result = run_novel(slug, novel_index)
        all_results.append(novel_result)
        write_json(ARTIFACT_ROOT / "all-novels-results.partial.json", all_results)
        print(
            f"{novel_result['status']}: test PPL "
            f"{novel_result['test_base_perplexity']:.2f} -> "
            f"{novel_result['test_trained_perplexity']:.2f}; "
            f"retention {novel_result['retention_regression_pct']:+.2f}%"
        )

    if len(all_results) != len(NOVELS) or {item["slug"] for item in all_results} != set(NOVELS):
        raise RuntimeError("The final ledger does not cover every required novel exactly once")

    missing_manifests = [
        slug
        for slug in NOVELS
        if not (ARTIFACT_ROOT / slug / "experiment-manifest.json").is_file()
    ]
    if missing_manifests:
        raise FileNotFoundError(f"Missing experiment manifests: {missing_manifests}")

    write_json(ARTIFACT_ROOT / "all-novels-results.json", all_results)
    results_frame = pd.DataFrame(all_results)
    results_frame.to_csv(ARTIFACT_ROOT / "all-novels-results.csv", index=False)
    (ARTIFACT_ROOT / "all-novels-results.partial.json").unlink(missing_ok=True)

    summary_columns = [
        "novel",
        "train_chapters",
        "validation_chapters",
        "test_chapters",
        "selected_step",
        "test_base_perplexity",
        "test_trained_perplexity",
        "test_improvement_pct",
        "bootstrap_ci95_lower_pct",
        "bootstrap_ci95_upper_pct",
        "retention_regression_pct",
        "elapsed_minutes",
        "peak_gpu_gib",
        "status",
    ]
    display(results_frame[summary_columns].round(3))
    print("\nStatus counts:")
    print(results_frame["status"].value_counts().to_string())
    print(f"\nArtifacts: {ARTIFACT_ROOT}")

In [ ]:
# Render the evidence only after all eight measured result rows exist.
if results_frame.empty:
    print("Evidence dashboard unavailable until the GPU experiment completes.")
else:
    import matplotlib.pyplot as plt

    dashboard = results_frame.sort_values("test_improvement_pct").reset_index(drop=True)
    positions = np.arange(len(dashboard))
    improvements = dashboard["test_improvement_pct"].to_numpy()
    lower = dashboard["bootstrap_ci95_lower_pct"].to_numpy()
    upper = dashboard["bootstrap_ci95_upper_pct"].to_numpy()
    error_bars = np.vstack([improvements - lower, upper - improvements])
    status_colors = {"PASS": "#2A9D8F", "INCONCLUSIVE": "#F2C14E", "FAIL": "#D1495B"}
    colors = [status_colors[status] for status in dashboard["status"]]

    figure, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].errorbar(
        improvements, positions, xerr=error_bars, fmt="none", ecolor=colors, capsize=4
    )
    axes[0].scatter(improvements, positions, c=colors, s=70, zorder=3)
    axes[0].axvline(0, color="black", linewidth=1)
    axes[0].axvline(DOMAIN_IMPROVEMENT_TARGET_PCT, color="#2A9D8F", linestyle="--")
    axes[0].set_yticks(positions, dashboard["novel"])
    axes[0].set_xlabel("Test perplexity improvement (%) with chapter CI")
    axes[0].set_title("Did Domain Evidence Clear Zero?", fontweight="bold")

    axes[1].scatter(
        dashboard["test_improvement_pct"],
        dashboard["retention_regression_pct"],
        c=colors,
        s=80,
    )
    axes[1].axvline(DOMAIN_IMPROVEMENT_TARGET_PCT, color="#2A9D8F", linestyle="--")
    axes[1].axhline(GENERAL_RETENTION_LIMIT_PCT, color="#D1495B", linestyle="--")
    axes[1].set_xlabel("Test perplexity improvement (%)")
    axes[1].set_ylabel("General-retention regression (%)")
    axes[1].set_title("Domain Gain vs Retention Cost", fontweight="bold")
    for row in dashboard.itertuples():
        axes[1].annotate(row.novel, (row.test_improvement_pct, row.retention_regression_pct), fontsize=7)

    runtime_points = axes[2].scatter(
        dashboard["elapsed_minutes"],
        dashboard["peak_gpu_gib"],
        c=colors,
        s=80,
    )
    axes[2].set_xlabel("Elapsed time (minutes)")
    axes[2].set_ylabel("Peak allocated GPU memory (GiB)")
    axes[2].set_title("What Did the Evidence Cost?", fontweight="bold")
    for row in dashboard.itertuples():
        axes[2].annotate(row.novel, (row.elapsed_minutes, row.peak_gpu_gib), fontsize=7)

    for axis in axes:
        axis.grid(alpha=0.2)
    figure.suptitle("Eight Independent LoRA CPT Experiments", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    display(
        dashboard[[
            "novel",
            "selected_step",
            "test_improvement_pct",
            "bootstrap_ci95_lower_pct",
            "bootstrap_ci95_upper_pct",
            "retention_regression_pct",
            "elapsed_minutes",
            "peak_gpu_gib",
            "status",
        ]].round(3)
    )

## 7. Read the evidence without pooling away the experiment

The result dashboard above the interpretation notes should be read in three passes: first ask whether each uncertainty interval clears zero, then check whether domain movement damaged general-language retention, and only then inspect runtime and memory. A fast run does not rescue failed evidence, and a large point estimate does not rescue an interval that crosses zero.

Validation, test, and retention answer different questions:

- **Validation NLL** selects one of the four step budgets for a single novel.
- **Test NLL/perplexity** measures that frozen choice on later, untouched chapters.
- **General-language retention** checks whether domain movement coincides with broad likelihood regression.
- **Matched generations** expose visible behavior but do not override the quantitative gates.

The chronological test block introduces real temporal and narrative-arc shift. That makes it a useful generalization challenge, but not an IID estimate of all possible passages. The chapter bootstrap has only four to six independent units, so interpret its interval as small-sample sensitivity to chapter membership.

Finally, each row comes from a different adapter trained from the same base. Do not pool all tokens into one apparent model score or treat chapters from one novel as independent replicates for another. Compare the per-novel gates, runtimes, and uncertainty intervals, then investigate failures at the chapter level in each manifest.